In [ ]:
import os
import random
import git
from pathlib import Path

import cv2
import numpy as np

# =======================
# GLOBAL PROJECT ROOT
# =======================
ROOT_DIR = Path(git.Repo(".", search_parent_directories=True).working_tree_dir)

# =======================
# CONFIG
# =======================
DATASET = "agriVision"
RAW_DATA_SUFFIX = "agriVision-RGB-cleaned-jitter"  # folders created: batch{idx}-{suffix}

SOURCE_DIR = ROOT_DIR / "raw-data" / DATASET / "uncleaned"
DATA_DIR   = ROOT_DIR / "raw-data" / DATASET

NUM_BATCHES = 10
FILES_PER_BATCH = 1500
SEED = 42

IMG_EXTS = (".jpg", ".jpeg", ".png")


# =======================
# HELPERS
# =======================
def list_files(folder):
    """Return sorted list of filenames (files only) filtered to image extensions."""
    return sorted(
        f for f in os.listdir(folder)
        if os.path.isfile(folder / f) and os.path.splitext(f)[1].lower() in IMG_EXTS
    )


def calculate_batch_avg_and_std_bgr(img_paths):
    """Compute (mean,std) for (B,G,R) over per-image channel means (OpenCV uses BGR)."""
    b_vals, g_vals, r_vals = [], [], []

    for p in img_paths:
        img = cv2.imread(p, cv2.IMREAD_COLOR)  # BGR uint8
        if img is None:
            continue
        b, g, r = cv2.split(img.astype(np.float32))
        b_vals.append(b.mean())
        g_vals.append(g.mean())
        r_vals.append(r.mean())

    if not b_vals:
        return (0.0, 0.0, 0.0), (1.0, 1.0, 1.0)

    mean = (float(np.mean(b_vals)), float(np.mean(g_vals)), float(np.mean(r_vals)))
    std = (
        float(np.std(b_vals) or 1.0),
        float(np.std(g_vals) or 1.0),
        float(np.std(r_vals) or 1.0),
    )
    return mean, std


def jitter_and_normalize_bgr(img_bgr, mean_bgr, std_bgr, rng):
    """Add small uniform noise then normalize channels using batch stats (BGR in/out)."""
    img = img_bgr.astype(np.float32)
    img += rng.uniform(-0.5, 0.5, size=img.shape).astype(np.float32)

    b, g, r = cv2.split(img)
    b_n = (b - mean_bgr[0]) / std_bgr[0]
    g_n = (g - mean_bgr[1]) / std_bgr[1]
    r_n = (r - mean_bgr[2]) / std_bgr[2]
    return cv2.merge([b_n, g_n, r_n])


# =======================
# MAIN
# =======================
def create_batches_and_jitter_npz(source_dir, data_dir, num_batches, files_per_batch, raw_data_suffix, seed=42):
    """Shuffle images into batches, then write jitter+normalized .npz saved as RGB."""
    files = list_files(source_dir)

    rng_py = random.Random(seed) if seed is not None else random.Random()
    rng_py.shuffle(files)

    rng_np = np.random.default_rng(seed)

    total_written = 0
    for batch_idx in range(num_batches):
        start = batch_idx * files_per_batch
        end = start + files_per_batch
        batch_files = files[start:end]

        out_folder = data_dir / f"batch{batch_idx}-{raw_data_suffix}"
        os.makedirs(out_folder, exist_ok=True)

        batch_paths = [str(source_dir / name) for name in batch_files]
        mean_bgr, std_bgr = calculate_batch_avg_and_std_bgr(batch_paths)

        written = 0
        for name in batch_files:
            img_path = str(source_dir / name)
            img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img_bgr is None:
                continue

            out_bgr = jitter_and_normalize_bgr(img_bgr, mean_bgr, std_bgr, rng_np)
            out_rgb = out_bgr[:, :, ::-1]  # convert once: BGR -> RGB for downstream

            base = os.path.splitext(name)[0]
            np.savez_compressed(str(out_folder / f"{base}.npz"), image=out_rgb)
            written += 1

        total_written += written
        print(f"batch{batch_idx}: {written} npz -> {out_folder}")

    print(f"Total written: {total_written}")


if __name__ == "__main__":
    create_batches_and_jitter_npz(
        source_dir=SOURCE_DIR,
        data_dir=DATA_DIR,
        num_batches=NUM_BATCHES,
        files_per_batch=FILES_PER_BATCH,
        raw_data_suffix=RAW_DATA_SUFFIX,
        seed=SEED,
    )


Files renamed successfully!
['V4EJX3X8K_1048-3815-1560-4327.npz', 'XNKGLTAMZ_1324-3860-1836-4372.npz', 'MVEJ4886I_3122-4418-3634-4930.npz', 'UQWBZ1G4Y_5569-2376-6081-2888.npz', '8EF9M4P6Y_13515-9079-14027-9591.npz']
